# NOTEBOOK THẬT CHI TIẾT — PHẦN VI: KOLLA-ANSIBLE DEPLOY OPENSTACK

Notebook này dùng cho **VII. PHẦN VI — KOLLA-ANSIBLE DEPLOY OPENSTACK**.

## Mô hình đang dùng

- **Controller VM 1:** `10.0.0.11`
- **Controller VM 2:** `10.0.0.12`
- **Controller VM 3:** `10.0.0.13`
- **Compute bare metal Server B:** `10.0.0.22`
- **Compute bare metal Server C:** `10.0.0.23`
- **Server A / stack1:** giữ `libvirtd` để chạy 3 Controller VM
- **Server B/C:** phải tắt `libvirtd` trước khi Kolla deploy

## Network chính

| Network | Dải IP | Vai trò |
|---|---:|---|
| br-mgmt | `10.0.0.0/24` | Kolla management, API internal, OVN tunnel |
| br-storage | `10.0.2.0/24` | OpenStack client traffic tới Ceph MON/OSD |
| br-cluster | `10.0.3.0/24` | Ceph OSD replication, không dùng cho OpenStack client |
| br-lab | `192.168.150.0/24` | SSH/debug/lab access |

## Lệnh SSH vào Controller VM 1

```bash
ssh -J ubuntu@192.168.150.9 ubuntu@10.0.0.11
```

## Yêu cầu trước khi chạy notebook

1. Ceph cluster đã bootstrap xong.
2. `ceph -s` ổn định.
3. Pool `images`, `volumes`, `vms`, `backups` đã tạo.
4. Keyring đã copy từ Server A vào `/tmp/` trên 3 Controller VM.
5. Kolla venv đã có sẵn tại `/opt/kolla-venv`.

# 0. Đăng nhập Controller VM 1

Chạy từ máy quản trị hoặc từ stack1.

In [ ]:
ssh -J ubuntu@192.168.150.9 ubuntu@10.0.0.11

Sau khi vào Controller VM 1, kiểm tra hostname và IP.

In [ ]:
hostname
ip -br addr
ip route

Kỳ vọng:

- Hostname: `controller-1`
- Có `br-mgmt` hoặc interface mgmt có IP `10.0.0.11/24`
- Ping được Controller VM 2/3 và Compute B/C qua `10.0.0.x`

In [ ]:
ping -c 3 10.0.0.12
ping -c 3 10.0.0.13
ping -c 3 10.0.0.22
ping -c 3 10.0.0.23

# BƯỚC 25 — Chuẩn bị `/etc/ceph/` và Kolla config

Chạy trong **Controller VM 1**.

## 25.1. Kích hoạt Kolla-Ansible virtualenv

In [ ]:
source /opt/kolla-venv/bin/activate

which kolla-ansible
kolla-ansible --version
ansible --version

Nếu không thấy `kolla-ansible`, kiểm tra lại quá trình tạo template Controller VM.

## 25.2. Tạo `/etc/kolla` và copy template config

Chạy trên Controller VM 1.

In [ ]:
sudo mkdir -p /etc/kolla
sudo chown -R ubuntu:ubuntu /etc/kolla

cp -r /opt/kolla-venv/share/kolla-ansible/etc_examples/kolla/* /etc/kolla/

ls -la /etc/kolla

Kỳ vọng có các file như:

- `globals.yml`
- `passwords.yml`

## 25.3. Tạo thư mục Ceph config cho Kolla

Các service cần Ceph config:

- Glance dùng pool `images`
- Cinder-volume dùng pool `volumes`
- Cinder-backup dùng pool `backups`
- Nova dùng pool `vms`

In [ ]:
sudo mkdir -p /etc/ceph

mkdir -p /etc/kolla/config/glance
mkdir -p /etc/kolla/config/cinder/cinder-volume
mkdir -p /etc/kolla/config/cinder/cinder-backup
mkdir -p /etc/kolla/config/nova

find /etc/kolla/config -maxdepth 3 -type d | sort

## 25.4. Kiểm tra Ceph files đã nằm trong `/tmp`

Các file này được copy từ Server A ở phần Bootstrap Ceph.

In [ ]:
ls -l /tmp/ceph.conf
ls -l /tmp/ceph.client.*.keyring

Nếu thiếu file, quay lại Server A và copy lại:

```bash
for vm in 10.0.0.11 10.0.0.12 10.0.0.13; do
  sudo scp /etc/ceph/ceph.conf /etc/ceph/ceph.client.*.keyring ubuntu@$vm:/tmp/
done
```

## 25.5. Copy Ceph files vào `/etc/ceph`

In [ ]:
sudo cp /tmp/ceph.conf /etc/ceph/
sudo cp /tmp/ceph.client.*.keyring /etc/ceph/

sudo ls -l /etc/ceph/

## 25.6. Copy Ceph config vào từng thư mục Kolla service

In [ ]:
# Glance
cp /etc/ceph/ceph.conf /etc/kolla/config/glance/
cp /etc/ceph/ceph.client.glance.keyring /etc/kolla/config/glance/

# Cinder volume
cp /etc/ceph/ceph.conf /etc/kolla/config/cinder/cinder-volume/
cp /etc/ceph/ceph.client.cinder.keyring /etc/kolla/config/cinder/cinder-volume/

# Cinder backup
cp /etc/ceph/ceph.conf /etc/kolla/config/cinder/cinder-backup/
cp /etc/ceph/ceph.client.cinder.keyring /etc/kolla/config/cinder/cinder-backup/
cp /etc/ceph/ceph.client.cinder-backup.keyring /etc/kolla/config/cinder/cinder-backup/

# Nova
cp /etc/ceph/ceph.conf /etc/kolla/config/nova/
cp /etc/ceph/ceph.client.nova.keyring /etc/kolla/config/nova/
cp /etc/ceph/ceph.client.cinder.keyring /etc/kolla/config/nova/

## 25.7. Verify cấu trúc file Ceph cho Kolla

In [ ]:
find /etc/kolla/config/ -name "*.keyring" -o -name "ceph.conf" | sort

Kỳ vọng thấy:

```text
/etc/kolla/config/cinder/cinder-backup/ceph.client.cinder-backup.keyring
/etc/kolla/config/cinder/cinder-backup/ceph.client.cinder.keyring
/etc/kolla/config/cinder/cinder-backup/ceph.conf
/etc/kolla/config/cinder/cinder-volume/ceph.client.cinder.keyring
/etc/kolla/config/cinder/cinder-volume/ceph.conf
/etc/kolla/config/glance/ceph.client.glance.keyring
/etc/kolla/config/glance/ceph.conf
/etc/kolla/config/nova/ceph.client.cinder.keyring
/etc/kolla/config/nova/ceph.client.nova.keyring
/etc/kolla/config/nova/ceph.conf
```

# BƯỚC 26 — Copy config sang Controller VM 2 và VM 3

Chạy trên **Controller VM 1**.

## 26.1. Test SSH sang Controller VM 2/3

In [ ]:
ssh ubuntu@10.0.0.12 "hostname && ip -br addr"
ssh ubuntu@10.0.0.13 "hostname && ip -br addr"

Nếu chưa SSH được, cần copy SSH key hoặc dùng password `123.abc` theo cloud-init.

## 26.2. Copy Ceph/Kolla config sang VM 2/3

In [ ]:
for vm in 10.0.0.12 10.0.0.13; do
  echo "===== Prepare $vm ====="

  ssh ubuntu@$vm 'sudo mkdir -p /etc/ceph &&     mkdir -p /etc/kolla/config/glance              /etc/kolla/config/cinder/cinder-volume              /etc/kolla/config/cinder/cinder-backup              /etc/kolla/config/nova &&     sudo chown -R ubuntu:ubuntu /etc/kolla/config'

  echo "===== Copy ceph files to $vm:/tmp ====="
  scp /etc/ceph/ceph.conf       /etc/ceph/ceph.client.*.keyring ubuntu@$vm:/tmp/

  echo "===== Install ceph files inside $vm ====="
  ssh ubuntu@$vm '
    sudo cp /tmp/ceph.conf /etc/ceph/
    sudo cp /tmp/ceph.client.*.keyring /etc/ceph/

    cp /etc/ceph/ceph.conf /etc/kolla/config/glance/
    cp /etc/ceph/ceph.client.glance.keyring /etc/kolla/config/glance/

    cp /etc/ceph/ceph.conf /etc/kolla/config/cinder/cinder-volume/
    cp /etc/ceph/ceph.client.cinder.keyring /etc/kolla/config/cinder/cinder-volume/

    cp /etc/ceph/ceph.conf /etc/kolla/config/cinder/cinder-backup/
    cp /etc/ceph/ceph.client.cinder.keyring /etc/kolla/config/cinder/cinder-backup/
    cp /etc/ceph/ceph.client.cinder-backup.keyring /etc/kolla/config/cinder/cinder-backup/

    cp /etc/ceph/ceph.conf /etc/kolla/config/nova/
    cp /etc/ceph/ceph.client.nova.keyring /etc/kolla/config/nova/
    cp /etc/ceph/ceph.client.cinder.keyring /etc/kolla/config/nova/
  '
done

## 26.3. Verify trên VM 2/3

In [ ]:
for vm in 10.0.0.12 10.0.0.13; do
  echo "===== $vm ====="
  ssh ubuntu@$vm 'find /etc/kolla/config/ -name "*.keyring" -o -name "ceph.conf" | sort'
done

# BƯỚC 27 — Tạo inventory và SSH key cho Ansible

Chạy trên **Controller VM 1**.

## 27.1. Tạo SSH key cho Ansible

Nếu file đã tồn tại, có thể bỏ qua hoặc backup trước.

In [ ]:
ls -l ~/.ssh/id_ed25519 ~/.ssh/id_ed25519.pub 2>/dev/null || true

ssh-keygen -t ed25519 -N "" -f ~/.ssh/id_ed25519

## 27.2. Copy SSH key tới tất cả node Kolla quản lý

Danh sách node:

- 3 controller VM: `10.0.0.11`, `10.0.0.12`, `10.0.0.13`
- 2 compute bare metal: `10.0.0.22`, `10.0.0.23`

Lưu ý: không copy tới `10.0.0.21` vì Server A không chạy compute trong mô hình mới.

In [ ]:
for ip in 10.0.0.11 10.0.0.12 10.0.0.13 10.0.0.22 10.0.0.23; do
  echo "===== ssh-copy-id $ip ====="
  ssh-copy-id -o StrictHostKeyChecking=no ubuntu@$ip
done

## 27.3. Test SSH không mật khẩu

In [ ]:
for ip in 10.0.0.11 10.0.0.12 10.0.0.13 10.0.0.22 10.0.0.23; do
  echo "===== $ip ====="
  ssh -o BatchMode=yes ubuntu@$ip "hostname"
done

## 27.4. Tạo inventory `/etc/kolla/multinode`

Chạy trên Controller VM 1.

In [ ]:
cat > /etc/kolla/multinode << 'EOF'
[control]
10.0.0.11 ansible_user=ubuntu
10.0.0.12 ansible_user=ubuntu
10.0.0.13 ansible_user=ubuntu

[network]

[compute]
# Server B và C bare metal — KHÔNG có libvirtd
10.0.0.22 ansible_user=ubuntu
10.0.0.23 ansible_user=ubuntu

[storage]
# cinder-volume/backup deploy trên Controller VM
10.0.0.11 ansible_user=ubuntu
10.0.0.12 ansible_user=ubuntu
10.0.0.13 ansible_user=ubuntu

[loadbalancer]
10.0.0.11 ansible_user=ubuntu
10.0.0.12 ansible_user=ubuntu
10.0.0.13 ansible_user=ubuntu

[monitoring]
10.0.0.11 ansible_user=ubuntu

[deployment]
localhost ansible_connection=local

EOF

cat /etc/kolla/multinode

## 27.5. Test Ansible ping

In [ ]:
ansible -i /etc/kolla/multinode all -m ping

Kỳ vọng: tất cả node trả về `pong`.

Nếu lỗi `sudo password`, kiểm tra user `ubuntu` có NOPASSWD sudo trên tất cả node.

# BƯỚC 28 — Tắt libvirtd trên Server B và Server C

⚠️ BẮT BUỘC:

- Server B/C là compute bare metal.
- Kolla sẽ deploy `nova_libvirt` container.
- Nếu `libvirtd` system service còn chạy thì conflict socket.

Không chạy bước này trên Server A.

## 28.1. Tắt libvirtd trên Server B/C bằng Ansible ad-hoc

Chạy từ Controller VM 1.

In [ ]:
ansible -i /etc/kolla/multinode compute -b -m shell -a '
systemctl stop libvirtd 2>/dev/null || true
systemctl stop libvirtd.socket 2>/dev/null || true
systemctl stop libvirtd-ro.socket 2>/dev/null || true
systemctl stop libvirtd-admin.socket 2>/dev/null || true
systemctl stop virtlogd 2>/dev/null || true

systemctl disable libvirtd 2>/dev/null || true
systemctl disable libvirtd.socket 2>/dev/null || true
systemctl disable libvirtd-ro.socket 2>/dev/null || true
systemctl disable libvirtd-admin.socket 2>/dev/null || true

rm -f /var/run/libvirt/libvirt-sock
rm -f /var/run/libvirt/libvirt-sock-ro
rm -f /var/run/libvirt/libvirt-admin-sock

rm -f /etc/openvswitch/conf.db
'

## 28.2. Verify libvirt socket đã gone trên B/C

In [ ]:
ansible -i /etc/kolla/multinode compute -b -m shell -a '
hostname
ls /var/run/libvirt/libvirt-sock 2>/dev/null && echo "STILL EXISTS" || echo "GONE"
systemctl is-active libvirtd 2>/dev/null || true
'

Kỳ vọng: `GONE`.

## 28.3. Xác nhận Server A vẫn còn libvirtd

Chạy trên **Server A / stack1**, không phải Controller VM.

In [ ]:
# Chạy trên Server A / stack1
sudo systemctl status libvirtd --no-pager
virsh list --all

Kỳ vọng:

- `libvirtd active (running)`
- Có `controller-vm-1`, `controller-vm-2`, `controller-vm-3`

# BƯỚC 29 — Cấu hình `/etc/kolla/globals.yml`

Chạy trên **Controller VM 1**.

## 29.1. Backup globals cũ nếu có

In [ ]:
cp /etc/kolla/globals.yml /etc/kolla/globals.yml.bak.$(date +%F-%H%M%S) 2>/dev/null || true

## 29.2. Ghi globals.yml chuẩn theo mô hình 1 Controller-host + 2 Compute

Lưu ý:

- `network_interface: br-mgmt`
- `neutron_external_interface: dummy0`
- `storage_interface: br-storage`
- `tunnel_interface: br-mgmt`
- `ceph_mon_nodes` dùng IP `br-storage`

In [ ]:
cat > /etc/kolla/globals.yml << 'EOF'
# ── Cơ bản ───────────────────────────────────────────────────
kolla_base_distro: ubuntu
kolla_install_type: binary
openstack_release: "2025.1"

# ── Network ──────────────────────────────────────────────────
# br-mgmt tồn tại trên cả Controller VM và Compute node
network_interface: "br-mgmt"

# dummy0 → Kolla tạo br-ex OVS bridge, gắn dummy0 vào
# Không tạo br-ex thủ công — Kolla tự làm
neutron_external_interface: "dummy0"

kolla_internal_vip_address: "10.0.0.10"
kolla_external_vip_address: "192.168.150.200"

# br-storage cho Ceph client traffic
storage_interface: "br-storage"

# br-mgmt cho OVN Geneve tunnel
tunnel_interface: "br-mgmt"

# ── Services ─────────────────────────────────────────────────
enable_openstack_core: "yes"
enable_glance: "yes"
enable_nova: "yes"
enable_neutron: "yes"
enable_cinder: "yes"
enable_horizon: "yes"
enable_heat: "no"
enable_swift: "no"

# ── Neutron OVN ──────────────────────────────────────────────
neutron_plugin_agent: "ovn"

# ── Coordination backend (bắt buộc với Ceph backend) ─────────
enable_valkey: "yes"

# ── Ceph HA setup ────────────────────────────────────────────
cinder_cluster_name: "cinder-cluster-1"

# ── Ceph backend ─────────────────────────────────────────────
glance_backend_ceph: "yes"
glance_backend_file: "no"
cinder_backend_ceph: "yes"
nova_backend_ceph: "yes"

ceph_glance_user: "glance"
ceph_glance_pool_name: "images"

ceph_cinder_user: "cinder"
ceph_cinder_pool_name: "volumes"

ceph_nova_user: "nova"
ceph_nova_pool_name: "vms"

# IP br-storage của Ceph MON — KHÔNG dùng hostname
ceph_mon_nodes: "10.0.2.21,10.0.2.22,10.0.2.23"

EOF

cat /etc/kolla/globals.yml

cat > /etc/kolla/globals.yml << EOF
# ── Cơ bản ─────────────────────────────────────────────────
kolla_base_distro: ubuntu
kolla_install_type: binary
openstack_release: "2025.2"

# ── Cấu hình Registry ───────────────────────────────────────
docker_registry: "quay.io"
docker_namespace: "openstack.kolla"

# ── Network interfaces ──────────────────────────────────────
network_interface: "br-mgmt"

# Server chỉ có 1 NIC vật lý, Kolla tạo br-ex và gắn dummy0
neutron_external_interface: "dummy0"
neutron_bridge_name: "br-ex"

kolla_internal_vip_address: "10.0.0.10"
kolla_external_vip_address: "192.168.150.200"

# Storage và tunnel
storage_interface: "br-storage"
tunnel_interface: "br-mgmt"

# ── Services ────────────────────────────────────────────────
enable_openstack_core: "yes"
enable_glance: "yes"
enable_nova: "yes"
enable_neutron: "yes"
enable_cinder: "yes"
enable_horizon: "yes"
enable_heat: "no"
enable_swift: "no"

# ── Neutron OVN ─────────────────────────────────────────────
neutron_plugin_agent: "ovn"

# ── Ceph backend ────────────────────────────────────────────
glance_backend_ceph: "yes"
glance_backend_file: "no"
cinder_backend_ceph: "yes"
nova_backend_ceph: "yes"

ceph_glance_user: "glance"
ceph_glance_pool_name: "images"

ceph_cinder_user: "cinder"
ceph_cinder_pool_name: "volumes"

ceph_nova_user: "nova"
ceph_nova_pool_name: "vms"

ceph_mon_nodes: "10.0.2.21,10.0.2.22,10.0.2.23"

# FSID lấy từ: grep fsid /etc/ceph/ceph.conf
ceph_cluster_fsid: "83b388c4-46aa-11f1-8a84-000c293450cf"

# Libvirt secret UUID cho Nova + Cinder
cinder_rbd_secret_uuid: "457eb676-33da-42ec-9a8c-9293d545c337"

# ── Coordination backend ───────────────────────────────────
enable_valkey: "yes"

# ── Cinder HA / Cluster ─────────────────────────────────────
cinder_cluster_name: "cinder-cluster-1"

# ── Nova compute ────────────────────────────────────────────
nova_compute_virt_type: "kvm"

# ── Cinder backup qua Ceph ──────────────────────────────────
enable_cinder_backup: "yes"
cinder_backup_driver: "ceph"
cinder_backup_backend_ceph: "yes"
ceph_cinder_backup_user: "cinder-backup"
ceph_cinder_backup_pool_name: "backups"
EOF

## 29.3. Kiểm tra các interface tồn tại trên node

Kolla yêu cầu `network_interface`, `storage_interface`, `tunnel_interface` tồn tại đúng tên trên node liên quan.

In [ ]:
ansible -i /etc/kolla/multinode all -m shell -a '
hostname
ip -br addr show br-mgmt || true
ip -br addr show br-storage || true
ip -br addr show dummy0 || true
'

Nếu `dummy0` chưa có trên compute/controller, tạo trước trên các node:

In [ ]:
ansible -i /etc/kolla/multinode all -b -m shell -a '
modprobe dummy 2>/dev/null || true
ip link add dummy0 type dummy 2>/dev/null || true
ip link set dummy0 up
ip -br link show dummy0
'

# BƯỚC 30 — Deploy OpenStack

Chạy trên **Controller VM 1** trong venv `/opt/kolla-venv`.

Khuyến nghị chạy trong `tmux` để tránh mất phiên SSH.

## 30.1. Mở tmux

In [ ]:
tmux new -s kolla-deploy

Nếu đã vào tmux rồi thì bỏ qua cell trên.

## 30.2. Active venv

In [ ]:
source /opt/kolla-venv/bin/activate
which kolla-ansible
kolla-ansible --version

## 30.3. Sinh passwords

In [ ]:
kolla-genpwd

grep -E "^keystone_admin_password:" /etc/kolla/passwords.yml || true

## 30.4. Set admin password dễ nhớ cho lab

In [ ]:
sed -i   "s/^keystone_admin_password:.*/keystone_admin_password: Lab@OpenStack2025!/"   /etc/kolla/passwords.yml

grep "^keystone_admin_password:" /etc/kolla/passwords.yml

## 30.5. Bootstrap servers

Bước này chuẩn bị node cho Kolla, cài/chỉnh Docker, Python dependencies, user, service.

In [ ]:
kolla-ansible -i /etc/kolla/multinode bootstrap-servers

Nếu lỗi ở compute B/C, kiểm tra:

```bash
ansible -i /etc/kolla/multinode compute -m ping
ansible -i /etc/kolla/multinode compute -b -m shell -a 'hostname; docker --version || true; ip -br addr'
```

## 30.6. Prechecks

Prechecks phải đạt **0 failed** trước khi deploy.

In [ ]:
kolla-ansible -i /etc/kolla/multinode prechecks

Nếu lỗi interface, kiểm tra lại:

```bash
ansible -i /etc/kolla/multinode all -m shell -a 'ip -br addr'
```

Nếu lỗi libvirt socket trên compute, quay lại Bước 28.

## 30.7. Pull images

Lần đầu có thể mất 30–60 phút.

In [ ]:
kolla-ansible -i /etc/kolla/multinode pull

## 30.8. Deploy OpenStack

In [ ]:
kolla-ansible -i /etc/kolla/multinode deploy

Nếu lỗi, chạy debug verbose:

```bash
kolla-ansible -i /etc/kolla/multinode deploy -vvv
```

## 30.9. Post-deploy

In [ ]:
kolla-ansible -i /etc/kolla/multinode post-deploy

## 30.10. Load admin credentials

In [ ]:
source /etc/kolla/admin-openrc.sh
openstack token issue

Kỳ vọng trả về token hợp lệ.

# Kiểm tra nhanh sau deploy

In [ ]:
source /etc/kolla/admin-openrc.sh

openstack service list
openstack compute service list
openstack network agent list
openstack volume service list

# Kiểm tra container trên các node

Chạy từ Controller VM 1.

In [ ]:
ansible -i /etc/kolla/multinode all -b -m shell -a '
hostname
docker ps --format "table {{.Names}}\t{{.Status}}" | head -n 30
'

# Kiểm tra Ceph backend từ OpenStack

Tạo thử volume 1GB.

In [ ]:
source /etc/kolla/admin-openrc.sh

openstack volume create --size 1 test-ceph-vol
sleep 10
openstack volume show test-ceph-vol
openstack volume delete test-ceph-vol

# Checklist hoàn thành phần Kolla deploy

- `openstack token issue` thành công.
- `openstack service list` có Keystone, Nova, Neutron, Glance, Cinder, Placement.
- `openstack compute service list` có nova-compute trên `10.0.0.22` và `10.0.0.23`.
- `openstack network agent list` có OVN agents alive.
- `openstack volume service list` cinder-volume up.
- Tạo volume test thành công và xóa được.

# Debug nhanh lỗi hay gặp

## 1. Inventory sai group

Kiểm tra:

```bash
cat /etc/kolla/multinode
```

Trong mô hình này, compute chỉ có:

```text
10.0.0.22
10.0.0.23
```

Không đưa `10.0.0.21` vào compute.

## 2. `libvirt-sock` còn tồn tại trên B/C

```bash
ansible -i /etc/kolla/multinode compute -b -m shell -a 'ls /var/run/libvirt/libvirt-sock && echo BAD || echo OK'
```

## 3. Sai Ceph keyring

```bash
find /etc/kolla/config/ -name "*.keyring" -o -name "ceph.conf" | sort
```

## 4. Interface không tồn tại

```bash
ansible -i /etc/kolla/multinode all -m shell -a 'ip -br addr'
```

## 5. Docker container exited

```bash
docker ps -a
docker logs <container_name>
```